# Notebook 06 — Préparation du dataset décisionnel Power BI

## Objectif

Ce notebook constitue la dernière étape du pipeline analytique.

Les traitements de qualité, de prétraitement, de construction des variables et de détection des anomalies ont été réalisés dans les notebooks précédents.

L'objectif ici est de :

- récupérer le résultat final ;
- effectuer les contrôles d'intégrité nécessaires ;
- conserver uniquement les informations utiles à l'analyse métier ;
- vérifier la cohérence du fichier destiné au reporting ;
- produire un dataset final stable et reproductible pour Power BI.


In [10]:
# =============================================================================
# Import des bibliothèques
# =============================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("Bibliothèques chargées.")

Bibliothèques chargées.


In [11]:
# =============================================================================
# Définition des chemins
# =============================================================================

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"

PROCESSED_DIR = DATA_DIR / "processed"

RESULTS_DIR = DATA_DIR / "results"

ANOMALY_DIR = RESULTS_DIR / "anomaly_detection"

ANOMALY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INPUT_PATH = ANOMALY_DIR / "anomaly_detection_powerbi.csv"

OUTPUT_PATH = ANOMALY_DIR / "anomaly_detection_powerbi.csv"


In [12]:
# =============================================================================
# Chargement du dataset issu de la détection des anomalies
# =============================================================================

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Dataset introuvable : {INPUT_PATH}"
    )

df = pd.read_csv(INPUT_PATH)

print("=" * 70)
print("DATASET ISSU DU NOTEBOOK PRECEDENT")
print("=" * 70)

print(f"Lignes     : {len(df):,}")
print(f"Colonnes   : {df.shape[1]:,}")
print(f"Taille     : {df.memory_usage(deep=True).sum()/1024**2:.2f} Mo")

DATASET ISSU DU NOTEBOOK PRECEDENT
Lignes     : 68,324
Colonnes   : 49
Taille     : 54.56 Mo


In [13]:
# =============================================================================
# Aperçu du dataset
# =============================================================================

display(df.head())

print("\nVariables disponibles :")
display(
    pd.DataFrame({
        "Variable": df.columns
    })
)

,Year,MonthLib,International Department,International Sub Department,International Family,EAN,Tx M3N MDH Inter N,Tx M3N non CP N,Tx M3N MDH Inter N-1,Tx M3N non CP N-1,Average Purchasing Price N,Average selling price N,Average Purchasing Price N-1,Average Selling Price N-1,Price_Gap_N,Price_Ratio_N,Purchase_Price_Variation,Selling_Price_Variation,Margin_MDH_Variation,Margin_Local_Variation,Family_Median_Selling_Price,Family_Median_Purchase_Price,Selling_Price_vs_Family,Purchase_Price_vs_Family,Family_Median_Margin,Margin_vs_Family,Family_Median_Margin_Local,Margin_Local_vs_Family,Business_Rule_Anomaly,Business_Rule_Reason,ML_Anomaly_Score,ML_Anomaly_Rank,ML_Top5pct,Anomaly_Signal,Business_Rule_And_ML,Anomaly_Reason,Anomaly_Priority,Temporal_Historical_Median_Selling_Price,Temporal_Historical_Median_Purchase_Price,Temporal_Historical_Median_Margin_MDH,Temporal_Historical_Median_Margin_Local,Temporal_Absolute_Deviation_Selling_Price,Temporal_Absolute_Deviation_Purchase_Price,Temporal_Absolute_Deviation_Margin_MDH,Temporal_Absolute_Deviation_Margin_Local,Family_Robust_Z_Selling_Price,Family_Robust_Z_Purchase_Price,Family_Robust_Z_Margin_MDH,Family_Robust_Z_Margin_Local
0,2025,January,12-Decoration,10-Textile,2488-Throws And Blankets,3014627554317,NaN,-85.6,NaN,NaN,8.614000,4.640000,NaN,NaN,-3.974000,0.538658,NaN,NaN,NaN,NaN,18.449091,10.952857,-13.809091,-2.338857,46.2,NaN,39.70,-125.30,True,Prix de vente inférieur au prix d'achat | Marg...,1.097077,31989.0,False,Règle métier,False,Règle métier : Prix de vente inférieur au prix...,Critique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.870922,-0.589449,NaN,-10.835089
1,2025,March,12-Decoration,10-Textile,2488-Throws And Blankets,3014627554317,NaN,-3.4,NaN,NaN,8.610000,8.330000,NaN,NaN,-0.280000,0.967480,NaN,NaN,NaN,NaN,18.903704,11.014304,-10.573704,-2.404304,46.3,NaN,43.35,-46.75,True,Prix de vente inférieur au prix d'achat | Marg...,1.011052,54750.0,False,Règle métier,False,Règle métier : Prix de vente inférieur au prix...,Critique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.290523,-0.612116,NaN,-3.776340
2,2025,January,12-Decoration,10-Textile,2488-Throws And Blankets,3014627554324,NaN,39.5,NaN,NaN,7.837083,12.944167,NaN,NaN,5.107083,1.651656,NaN,NaN,NaN,NaN,18.449091,10.952857,-5.504924,-3.115774,46.2,NaN,39.70,-0.20,False,NaN,1.288898,13483.0,False,Aucun signal,False,Aucune information explicative disponible,Normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.745833,-0.785251,NaN,-0.017295
3,2025,April,12-Decoration,10-Textile,2488-Throws And Blankets,3014627554324,NaN,52.7,NaN,NaN,7.835000,16.580000,NaN,NaN,8.745000,2.116146,NaN,NaN,NaN,NaN,18.322778,10.852000,-1.742778,-3.017000,46.8,NaN,41.70,11.00,False,NaN,1.131364,26683.0,False,Aucun signal,False,Aucune information explicative disponible,Normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.230080,-0.899438,NaN,0.776900
4,2025,August,12-Decoration,10-Textile,2488-Throws And Blankets,3014627554324,NaN,52.7,NaN,NaN,7.840000,16.580000,NaN,NaN,8.740000,2.114796,NaN,NaN,NaN,NaN,19.282143,10.347500,-2.702143,-2.507500,48.7,NaN,46.20,6.50,False,NaN,1.044596,43641.0,False,Aucun signal,False,Aucune information explicative disponible,Normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.349724,-0.619397,NaN,0.584559



Variables disponibles :


,Variable
0,Year
1,MonthLib
2,International Department
3,International Sub Department
4,International Family
5,EAN
6,Tx M3N MDH Inter N
7,Tx M3N non CP N
8,Tx M3N MDH Inter N-1
9,Tx M3N non CP N-1


In [14]:
# =============================================================================
# Harmonisation du calendrier
# =============================================================================

MONTH_ORDER = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

unknown_months = sorted(
    set(df["MonthLib"].dropna().astype(str)) - set(MONTH_ORDER)
)

if unknown_months:
    raise ValueError(
        f"Valeurs MonthLib non reconnues : {unknown_months}"
    )

if df["MonthLib"].isna().any():
    raise ValueError("MonthLib contient des valeurs manquantes.")

df["Month"] = df["MonthLib"].map(MONTH_ORDER).astype("Int64")



In [15]:
# =============================================================================
# Variables indispensables au reporting Power BI
# =============================================================================

required_columns = [

    # =========================================================================
    # 1. TEMPS
    # =========================================================================

    "Year",
    "Month",
    "MonthLib",

    # =========================================================================
    # 2. HIERARCHIE METIER
    # =========================================================================

    "International Department",
    "International Sub Department",
    "International Family",
    "EAN",

    # =========================================================================
    # 3. VALEURS ECONOMIQUES
    # =========================================================================

    "Tx M3N MDH Inter N",
    "Tx M3N non CP N",
    "Tx M3N MDH Inter N-1",
    "Tx M3N non CP N-1",

    "Average Purchasing Price N",
    "Average selling price N",
    "Average Purchasing Price N-1",
    "Average Selling Price N-1",

    # =========================================================================
    # 4. DETECTION — REGLES METIER
    # =========================================================================

    "Business_Rule_Anomaly",
    "Business_Rule_Reason",

    # =========================================================================
    # 5. MACHINE LEARNING
    # =========================================================================

    "ML_Anomaly_Score",
    "ML_Anomaly_Rank",
    "ML_Top5pct",

    # =========================================================================
    # 6. SIGNAUX ET RESULTAT FINAL
    # =========================================================================

    "Anomaly_Signal",
    "Business_Rule_And_ML",

    "Anomaly_Reason",
    "Anomaly_Priority"
]

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_columns:

    raise ValueError(
        "Colonnes indispensables absentes : "
        f"{missing_columns}"
    )

print("Toutes les variables indispensables sont présentes.")

Toutes les variables indispensables sont présentes.


In [16]:
# =============================================================================
# Variables destinées au reporting Power BI
# =============================================================================

bi_columns = [

    # =========================================================================
    # 1. TEMPS
    # =========================================================================

    "Year",
    "Month",
    "MonthLib",

    # =========================================================================
    # 2. HIERARCHIE METIER
    # =========================================================================

    "International Department",
    "International Sub Department",
    "International Family",
    "EAN",

    # =========================================================================
    # 3. VALEURS ECONOMIQUES SOURCES
    # =========================================================================

    "Tx M3N MDH Inter N",
    "Tx M3N non CP N",

    "Tx M3N MDH Inter N-1",
    "Tx M3N non CP N-1",

    "Average Purchasing Price N",
    "Average selling price N",

    "Average Purchasing Price N-1",
    "Average Selling Price N-1",

    # =========================================================================
    # 4. VARIABLES ECONOMIQUES CALCULEES
    # =========================================================================

    "Price_Gap_N",
    "Price_Ratio_N",


    "Purchase_Price_Variation",
    "Selling_Price_Variation",


    "Margin_MDH_Variation",
    "Margin_Local_Variation",

    # =========================================================================
    # 5. COMPARAISON FAMILLE
    # =========================================================================

    "Family_Median_Selling_Price",
    "Family_Median_Purchase_Price",

    "Selling_Price_vs_Family",
    "Purchase_Price_vs_Family",

    "Family_Median_Margin",
    "Margin_vs_Family",

    "Family_Median_Margin_Local",
    "Margin_Local_vs_Family",

    # =========================================================================
    # 6. REGLES METIER
    # =========================================================================

    "Business_Rule_Anomaly",
    "Business_Rule_Reason",

    # =========================================================================
    # 7. CONTEXTE HISTORIQUE
    # =========================================================================

    "Temporal_Historical_Median_Selling_Price",
    "Temporal_Historical_Median_Purchase_Price",
    "Temporal_Historical_Median_Margin_MDH",
    "Temporal_Historical_Median_Margin_Local",

    "Temporal_Absolute_Deviation_Selling_Price",
    "Temporal_Absolute_Deviation_Purchase_Price",
    "Temporal_Absolute_Deviation_Margin_MDH",
    "Temporal_Absolute_Deviation_Margin_Local",

    # =========================================================================
    # 8. ROBUST Z — CONTEXTE FAMILLE
    # =========================================================================

    "Family_Robust_Z_Selling_Price",
    "Family_Robust_Z_Purchase_Price",
    "Family_Robust_Z_Margin_MDH",
    "Family_Robust_Z_Margin_Local",

    # =========================================================================
    # 9. MACHINE LEARNING
    # =========================================================================

    "ML_Anomaly_Score",
    "ML_Anomaly_Rank",
    "ML_Top5pct",

    # =========================================================================
    # 10. RESULTAT FINAL
    # =========================================================================

    "Anomaly_Signal",
    "Business_Rule_And_ML",

    "Anomaly_Reason",
    "Anomaly_Priority"
]

missing_bi_columns = [
    col
    for col in bi_columns
    if col not in df.columns
]

if missing_bi_columns:

    raise ValueError(
        "Colonnes nécessaires absentes du dataset : "
        f"{missing_bi_columns}"
    )

df_powerbi = df.loc[:, bi_columns].copy()

print("=" * 70)
print("DATASET POWER BI")
print("=" * 70)

print(f"Lignes     : {len(df_powerbi):,}")
print(f"Colonnes   : {df_powerbi.shape[1]:,}")

display(df_powerbi.head())

DATASET POWER BI
Lignes     : 68,324
Colonnes   : 50


,Year,Month,MonthLib,International Department,International Sub Department,International Family,EAN,Tx M3N MDH Inter N,Tx M3N non CP N,Tx M3N MDH Inter N-1,Tx M3N non CP N-1,Average Purchasing Price N,Average selling price N,Average Purchasing Price N-1,Average Selling Price N-1,Price_Gap_N,Price_Ratio_N,Purchase_Price_Variation,Selling_Price_Variation,Margin_MDH_Variation,Margin_Local_Variation,Family_Median_Selling_Price,Family_Median_Purchase_Price,Selling_Price_vs_Family,Purchase_Price_vs_Family,Family_Median_Margin,Margin_vs_Family,Family_Median_Margin_Local,Margin_Local_vs_Family,Business_Rule_Anomaly,Business_Rule_Reason,Temporal_Historical_Median_Selling_Price,Temporal_Historical_Median_Purchase_Price,Temporal_Historical_Median_Margin_MDH,Temporal_Historical_Median_Margin_Local,Temporal_Absolute_Deviation_Selling_Price,Temporal_Absolute_Deviation_Purchase_Price,Temporal_Absolute_Deviation_Margin_MDH,Temporal_Absolute_Deviation_Margin_Local,Family_Robust_Z_Selling_Price,Family_Robust_Z_Purchase_Price,Family_Robust_Z_Margin_MDH,Family_Robust_Z_Margin_Local,ML_Anomaly_Score,ML_Anomaly_Rank,ML_Top5pct,Anomaly_Signal,Business_Rule_And_ML,Anomaly_Reason,Anomaly_Priority
0,2025,1,January,12-Decoration,10-Textile,2488-Throws And Blankets,3014627554317,NaN,-85.6,NaN,NaN,8.614000,4.640000,NaN,NaN,-3.974000,0.538658,NaN,NaN,NaN,NaN,18.449091,10.952857,-13.809091,-2.338857,46.2,NaN,39.70,-125.30,True,Prix de vente inférieur au prix d'achat | Marg...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.870922,-0.589449,NaN,-10.835089,1.097077,31989.0,False,Règle métier,False,Règle métier : Prix de vente inférieur au prix...,Critique
1,2025,3,March,12-Decoration,10-Textile,2488-Throws And Blankets,3014627554317,NaN,-3.4,NaN,NaN,8.610000,8.330000,NaN,NaN,-0.280000,0.967480,NaN,NaN,NaN,NaN,18.903704,11.014304,-10.573704,-2.404304,46.3,NaN,43.35,-46.75,True,Prix de vente inférieur au prix d'achat | Marg...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.290523,-0.612116,NaN,-3.776340,1.011052,54750.0,False,Règle métier,False,Règle métier : Prix de vente inférieur au prix...,Critique
2,2025,1,January,12-Decoration,10-Textile,2488-Throws And Blankets,3014627554324,NaN,39.5,NaN,NaN,7.837083,12.944167,NaN,NaN,5.107083,1.651656,NaN,NaN,NaN,NaN,18.449091,10.952857,-5.504924,-3.115774,46.2,NaN,39.70,-0.20,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.745833,-0.785251,NaN,-0.017295,1.288898,13483.0,False,Aucun signal,False,Aucune information explicative disponible,Normal
3,2025,4,April,12-Decoration,10-Textile,2488-Throws And Blankets,3014627554324,NaN,52.7,NaN,NaN,7.835000,16.580000,NaN,NaN,8.745000,2.116146,NaN,NaN,NaN,NaN,18.322778,10.852000,-1.742778,-3.017000,46.8,NaN,41.70,11.00,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.230080,-0.899438,NaN,0.776900,1.131364,26683.0,False,Aucun signal,False,Aucune information explicative disponible,Normal
4,2025,8,August,12-Decoration,10-Textile,2488-Throws And Blankets,3014627554324,NaN,52.7,NaN,NaN,7.840000,16.580000,NaN,NaN,8.740000,2.114796,NaN,NaN,NaN,NaN,19.282143,10.347500,-2.702143,-2.507500,48.7,NaN,46.20,6.50,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.349724,-0.619397,NaN,0.584559,1.044596,43641.0,False,Aucun signal,False,Aucune information explicative disponible,Normal


In [17]:
# =============================================================================
# Contrôle de structure du dataset Power BI
# =============================================================================

print("=" * 70)
print("CONTROLE DE STRUCTURE DU DATASET POWER BI")
print("=" * 70)

df_powerbi["EAN"] = df_powerbi["EAN"].astype("string")

boolean_columns = [
    "Business_Rule_Anomaly",
    "ML_Top5pct",
    "Business_Rule_And_ML"
]

for col in boolean_columns:
    df_powerbi[col] = (
        df_powerbi[col]
        .fillna(False)
        .astype(bool)
    )

print(f"Nombre de lignes   : {len(df_powerbi):,}")
print(f"Nombre de colonnes : {df_powerbi.shape[1]:,}")

print("\nTypes des signaux :")
display(df_powerbi[boolean_columns].dtypes.to_frame("Type"))


CONTROLE DE STRUCTURE DU DATASET POWER BI
Nombre de lignes   : 68,324
Nombre de colonnes : 50

Types des signaux :


,Type
Business_Rule_Anomaly,bool
ML_Top5pct,bool
Business_Rule_And_ML,bool


In [18]:
# =============================================================================
# Contrôles des identifiants
# =============================================================================

print("=" * 70)
print("CONTROLES D'IDENTIFICATION")
print("=" * 70)

print(f"EAN manquants : {df_powerbi['EAN'].isna().sum():,}")
print(f"EAN uniques   : {df_powerbi['EAN'].nunique():,}")
print(f"Doublons exacts : {df_powerbi.duplicated().sum():,}")

key_duplicates = (
    df_powerbi
    .duplicated(subset=["Year", "Month", "EAN"])
    .sum()
)

print(f"Doublons EAN × mois : {key_duplicates:,}")

if key_duplicates > 0:
    raise ValueError(
        "Le dataset final contient des doublons sur la clé Year × Month × EAN."
    )

if df_powerbi["EAN"].isna().any():
    raise ValueError(
        "Le dataset final contient des EAN manquants."
    )


CONTROLES D'IDENTIFICATION
EAN manquants : 0
EAN uniques   : 5,781
Doublons exacts : 0
Doublons EAN × mois : 0


In [19]:
# =============================================================================
# Contrôle des variables d'anomalie
# =============================================================================

print("=" * 70)
print("CONTROLES DES ANOMALIES")
print("=" * 70)

rule_count = int(df_powerbi["Business_Rule_Anomaly"].sum())
ml_count = int(df_powerbi["ML_Top5pct"].sum())
double_count = int(df_powerbi["Business_Rule_And_ML"].sum())

critical_count = int(df_powerbi["Anomaly_Priority"].eq("Critique").sum())
monitoring_count = int(df_powerbi["Anomaly_Priority"].eq("À surveiller").sum())
normal_count = int(df_powerbi["Anomaly_Priority"].eq("Normal").sum())

print("Anomalies règle métier :", rule_count)
print("Observations Top 5 % ML :", ml_count)
print("Doubles signaux règle métier + ML :", double_count)
print("Observations Critiques :", critical_count)
print("Observations À surveiller :", monitoring_count)
print("Observations Normales :", normal_count)

if critical_count + monitoring_count + normal_count != len(df_powerbi):
    raise ValueError(
        "La répartition des priorités ne couvre pas toute la population."
    )

if double_count > min(rule_count, ml_count):
    raise ValueError(
        "Le nombre de doubles signaux est incohérent avec les signaux individuels."
    )


CONTROLES DES ANOMALIES
Anomalies règle métier : 1663
Observations Top 5 % ML : 3417
Doubles signaux règle métier + ML : 316
Observations Critiques : 1663
Observations À surveiller : 3101
Observations Normales : 63560


In [20]:
# =============================================================================
# Contrôle final de la population
# =============================================================================

print("=" * 70)
print("CONTROLE FINAL DE LA POPULATION")
print("=" * 70)

print(f"Nombre de lignes EAN × mois : {len(df_powerbi):,}")
print(f"Nombre d'EAN : {df_powerbi['EAN'].nunique():,}")

priority_counts = (
    df_powerbi["Anomaly_Priority"]
    .value_counts(dropna=False)
)

print("\nRépartition des priorités :")
display(priority_counts.to_frame("Nombre"))

critical_count = int(df_powerbi["Anomaly_Priority"].eq("Critique").sum())
monitoring_count = int(df_powerbi["Anomaly_Priority"].eq("À surveiller").sum())
normal_count = int(df_powerbi["Anomaly_Priority"].eq("Normal").sum())

print(f"Observations Critiques    : {critical_count:,}")
print(f"Observations À surveiller : {monitoring_count:,}")
print(f"Observations Normales     : {normal_count:,}")

print(f"Taux Critique : {critical_count / len(df_powerbi):.2%}")
print(f"Taux À surveiller : {monitoring_count / len(df_powerbi):.2%}")
print(f"Taux Normal : {normal_count / len(df_powerbi):.2%}")

if not np.isclose(
    (critical_count + monitoring_count + normal_count) / len(df_powerbi),
    1.0
):
    raise ValueError("Les taux de priorité ne couvrent pas 100 % de la population.")


CONTROLE FINAL DE LA POPULATION
Nombre de lignes EAN × mois : 68,324
Nombre d'EAN : 5,781

Répartition des priorités :


,Nombre
Anomaly_Priority,
Normal,63560
À surveiller,3101
Critique,1663


Observations Critiques    : 1,663
Observations À surveiller : 3,101
Observations Normales     : 63,560
Taux Critique : 2.43%
Taux À surveiller : 4.54%
Taux Normal : 93.03%


In [21]:
# =============================================================================
# Contrôle des signaux et priorités
# =============================================================================

print("=" * 70)
print("PRIORITES")
print("=" * 70)

display(
    df_powerbi["Anomaly_Priority"]
    .value_counts(dropna=False)
    .to_frame("Nombre")
)

print("=" * 70)
print("SIGNAUX")
print("=" * 70)

display(
    df_powerbi["Anomaly_Signal"]
    .value_counts(dropna=False)
    .to_frame("Nombre")
)

print("=" * 70)
print("DOUBLE SIGNAL METIER + ML")
print("=" * 70)

display(
    df_powerbi["Business_Rule_And_ML"]
    .value_counts(dropna=False)
    .to_frame("Nombre")
)

required_final_fields = [
    "Anomaly_Reason",
    "Anomaly_Priority",
    "Anomaly_Signal",
    "Business_Rule_And_ML"
]

missing_final_values = {
    col: int(df_powerbi[col].isna().sum())
    for col in required_final_fields
    if df_powerbi[col].isna().any()
}

if missing_final_values:
    raise ValueError(
        "Valeurs manquantes dans les champs de restitution : "
        f"{missing_final_values}"
    )

print("\nContrôle des champs de restitution : OK")


PRIORITES


,Nombre
Anomaly_Priority,
Normal,63560
À surveiller,3101
Critique,1663


SIGNAUX


,Nombre
Anomaly_Signal,
Aucun signal,63560
ML score,3101
Règle métier,1347
Règle métier | ML score,316


DOUBLE SIGNAL METIER + ML


,Nombre
Business_Rule_And_ML,
False,68008
True,316



Contrôle des champs de restitution : OK


In [22]:
# =============================================================================
# Export final Power BI
# =============================================================================

df_powerbi.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("EXPORT POWER BI TERMINE")
print("=" * 70)

print(f"Lignes  : {len(df_powerbi):,}")
print(f"Colonnes : {df_powerbi.shape[1]:,}")


EXPORT POWER BI TERMINE
Lignes  : 68,324
Colonnes : 50


In [24]:
# =============================================================================
# Vérification du fichier exporté
# =============================================================================

if not OUTPUT_PATH.exists():
    raise FileNotFoundError(
        "Le fichier final n'a pas été créé."
    )

df_check = pd.read_csv(OUTPUT_PATH)

print("=" * 70)
print("VERIFICATION DE L'EXPORT")
print("=" * 70)

print(f"Lignes exportées : {len(df_check):,}")
print(f"Colonnes exportées : {len(df_check.columns):,}")

rows_ok = len(df_check) == len(df_powerbi)
columns_ok = list(df_check.columns) == list(df_powerbi.columns)

print("Correspondance du nombre de lignes :", rows_ok)
print("Correspondance des colonnes :", columns_ok)

if not rows_ok or not columns_ok:
    raise ValueError(
        "Le fichier exporté ne correspond pas au dataset Power BI préparé."
    )

print("Export du fichier avec succès.")


VERIFICATION DE L'EXPORT
Lignes exportées : 68,324
Colonnes exportées : 50
Correspondance du nombre de lignes : True
Correspondance des colonnes : True
Export du fichier avec succès.
